## Import et config

In [9]:
import os
import random
import warnings
from copy import deepcopy

import numpy as np
import pandas as pd
import scipy.io
import matplotlib.pyplot as plt
import seaborn as sns

from IPython.display import display

from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

warnings.filterwarnings("ignore")

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (12, 5)

SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)

try:
    tf.config.experimental.enable_op_determinism()
except Exception:
    pass

print(f"Seed globale: {SEED}")

CONFIG = {
    "mat_file_path": r"../data/Données brutes/DataPhase1.mat",
    "mat_variable": "DataPhase1",
    "selected_columns": {
        5: "repos",
        6: "environnement_immersif",
        7: "coherence_cardiaque_immersif",
    },
    "split_method": "random",   # "group" ou "random"
    "test_size": 0.20,
    "val_size": 0.20,
    "stride": 1,
    "clip_quantiles": (0.01, 0.99),
    "noise_std_robustness": 0.05,
    "noise_repeats": 5,
    "max_epochs": 30,
    "patience": 5,
}

CLASS_NAMES = list(CONFIG["selected_columns"].values())
NUM_CLASSES = len(CLASS_NAMES)

print("Configuration chargée.")
print(pd.Series(CONFIG))

Seed globale: 42
Configuration chargée.
mat_file_path                       ../data/Données brutes/DataPhase1.mat
mat_variable                                                   DataPhase1
selected_columns        {5: 'repos', 6: 'environnement_immersif', 7: '...
split_method                                                       random
test_size                                                             0.2
val_size                                                              0.2
stride                                                                  1
clip_quantiles                                               (0.01, 0.99)
noise_std_robustness                                                 0.05
noise_repeats                                                           5
max_epochs                                                             30
patience                                                                5
dtype: object


## Prétraitement des données

In [10]:
# A décaler dans un fichier .py pour éviter les redondances
def extract_workspace_strings(mat_dict):
    """Extrait les chaînes UTF-16LE depuis le workspace MATLAB embarqué.
    
    Retourne une liste ordonnée de chaînes lisibles (longueur >= 2),
    telles que les identifiants sujets et les noms de colonnes.
    """
    ws = mat_dict.get('__function_workspace__')
    if ws is None:
        return []
    raw = ws.tobytes()
    strings = []
    i = 0
    while i < len(raw) - 1:
        if raw[i + 1] == 0 and 32 <= raw[i] <= 126:
            end = i
            while end < len(raw) - 1 and raw[end + 1] == 0 and 32 <= raw[end] <= 126:
                end += 2
            if end - i >= 4:  # au moins 2 caractères
                s = raw[i:end].decode('utf-16le', errors='replace')
                strings.append(s)
            i = end + 2
        else:
            i += 1
    return strings


def get_subject_ids(mat_dict, n_subjects=42):
    """Extrait les identifiants sujets depuis le workspace MATLAB.
    
    Les identifiants apparaissent en premier dans la liste des chaînes UTF-16LE.
    """
    strings = extract_workspace_strings(mat_dict)
    # Filtrer les chaînes parasites (caractères spéciaux) et garder seulement les IDs
    subject_ids = []
    seen = set()
    for s in strings:
        if len(s) >= 2 and all(ord(c) < 128 for c in s) and s not in seen:
            # Heuristique : les IDs sujets sont alphanumériques et arrivent avant les noms de colonnes
            seen.add(s)
            subject_ids.append(s)
            if len(subject_ids) == n_subjects:
                break
    return subject_ids


def cell_to_float(cell):
    """Convertit une cellule MATLAB en float (NaN si vide ou MCOS)."""
    if isinstance(cell, (int, float)):
        return float(cell)
    if hasattr(cell, 'shape'):
        if cell.size == 0:
            return np.nan
        # MCOS opaque
        if cell.dtype.names and 's1' in cell.dtype.names:
            return np.nan
        return float(cell.flat[0])
    return np.nan


print('Fonctions utilitaires chargées.')

Fonctions utilitaires chargées.


In [ ]:
mat_path = CONFIG["mat_file_path"]
mat_var = CONFIG["mat_variable"]

if not os.path.exists(mat_path):
    raise FileNotFoundError(f"Fichier introuvable : {mat_path}")

mat_dict = scipy.io.loadmat(mat_path)

print(f"Fichier chargé : {mat_path}")
print(f"Clés disponibles : {list(mat_dict.keys())}")

if mat_var not in mat_dict:
    raise KeyError(f"Variable '{mat_var}' absente du fichier .mat")

data_matrix = np.asarray(mat_dict[mat_var], dtype=object)

print(f"Variable principale : {mat_var}")
print(f"Shape brute : {data_matrix.shape}")

subject_ids = get_subject_ids(mat_dict)
print(f"Nombre d'identifiants sujets extraits : {len(subject_ids)}")
print("Aperçu IDs sujets :", subject_ids[:10])

workspace_strings = extract_workspace_strings(mat_dict)
print(f"Nombre de chaînes extraites du workspace : {len(workspace_strings)}")
print("Aperçu chaînes workspace :", workspace_strings[:20])

# Voir colonnes
display(pd.DataFrame({
    "workspace_strings": workspace_strings
}))

Fichier chargé : ../data/Données brutes/DataPhase1.mat
Clés disponibles : ['__header__', '__version__', '__globals__', 'DataPhase1', '__function_workspace__']
Variable principale : DataPhase1
Shape brute : (42, 4)
Nombre d'identifiants sujets extraits : 42
Aperçu IDs sujets : ['136KD7', '0933', '7180bt', '22009611', 'ab0710', 'am3001', 'AP1133', 'BD1510', 'BM1997', 'CC1969']
Nombre de chaînes extraites du workspace : 15451
Aperçu chaînes workspace : ['136KD7', '0933', '7180bt', '22009611', 'ab0710', 'am3001', 'AP1133', 'BD1510', 'BM1997', 'CC1969', 'CH1308', 'CL1901', 'CM8976', 'CP1234', 'DZ1233', 'EL5421', 'EM1912', 'FC1997', 'GE2247', 'GG34']


,workspace_strings
0,136KD7
1,0933
2,7180bt
3,22009611
4,ab0710
...,...
15446,Y gaze direction
15447,Confidence
15448,isBoat
15449,X World Position


## Définition colonnes utiles

In [12]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.model_selection import ParameterGrid

FEATURE_COLUMNS = {
    0: "Time",
    1: "HMDPosX",
    2: "HMDPosY",
    3: "HMDPosZ",
    4: "RotX",
    5: "RotY",
    6: "RotZ",
    7: "SuggestedRotationX",
    8: "SuggestedRotationY",
    9: "SuggestedRotationZ",
    10: "LeftPupilDiameter",
    11: "RightPupilDiameter",
    12: "XGazeDirection",
    13: "YGazeDirection",
    14: "Confidence",
    15: "IsBoat",
    16: "XWorldPosition",
    17: "YWorldPosition",
}

AGGREGATED_FEATURE_COLUMNS = {
    k: v for k, v in FEATURE_COLUMNS.items() if k != 0
}

print("Colonnes d'entrée définies pour l'approche A (RandomForest).")
display(pd.DataFrame({
    "col_idx": list(FEATURE_COLUMNS.keys()),
    "feature_name": list(FEATURE_COLUMNS.values())
}))

Colonnes d'entrée définies pour l'approche A (RandomForest).


,col_idx,feature_name
0,0,Time
1,1,HMDPosX
2,2,HMDPosY
3,3,HMDPosZ
4,4,RotX
5,5,RotY
6,6,RotZ
7,7,SuggestedRotationX
8,8,SuggestedRotationY
9,9,SuggestedRotationZ


## Construction de la représentation

In [ ]:
def to_numeric_series(cell):
    arr = np.asarray(cell).squeeze()
    if arr.size == 0:
        return None
    try:
        arr = np.asarray(arr, dtype=float).reshape(-1)
    except (TypeError, ValueError):
        return None
    arr = arr[~np.isnan(arr)]
    return arr if arr.size > 0 else None


def compute_series_stats(values, time_values=None):
    values = np.asarray(values, dtype=float).reshape(-1)
    out = {
        "mean": float(np.mean(values)),
        "std": float(np.std(values)),
        "min": float(np.min(values)),
        "max": float(np.max(values)),
        "median": float(np.median(values)),
        "q25": float(np.quantile(values, 0.25)),
        "q75": float(np.quantile(values, 0.75)),
    }

    if len(values) >= 2:
        if time_values is not None and len(time_values) == len(values):
            x = np.asarray(time_values, dtype=float)
        else:
            x = np.arange(len(values), dtype=float)

        if np.unique(x).size >= 2:
            out["slope"] = float(np.polyfit(x, values, 1)[0])
        else:
            out["slope"] = 0.0
    else:
        out["slope"] = 0.0

    return out


def build_aggregated_dataset(data_matrix, feature_columns, subject_ids=None):
    rows = []

    for row_idx in range(data_matrix.shape[0]):
        row = {
            "row_id": int(row_idx),
            "subject_id": subject_ids[row_idx] if row_idx < len(subject_ids) else f"subject_{row_idx:03d}",
        }

        time_values = None
        if 0 in FEATURE_COLUMNS and 0 < data_matrix.shape[1]:
            time_values = to_numeric_series(data_matrix[row_idx, 0])

        valid_features = 0

        for col_idx, feature_name in feature_columns.items():
            if col_idx >= data_matrix.shape[1]:
                continue

            series = to_numeric_series(data_matrix[row_idx, col_idx])
            if series is None:
                continue

            stats = compute_series_stats(series, time_values=time_values)
            for stat_name, stat_value in stats.items():
                row[f"{feature_name}__{stat_name}"] = stat_value

            valid_features += 1

        row["n_valid_features"] = valid_features
        rows.append(row)

    return pd.DataFrame(rows)


features_df = build_aggregated_dataset(
    data_matrix=data_matrix,
    feature_columns=AGGREGATED_FEATURE_COLUMNS,
    subject_ids=subject_ids
)

print("Jeu tabulaire agrégé construit.")
print(f"Shape: {features_df.shape}")
display(features_df.head())

missing_ratio = features_df.isna().mean().sort_values(ascending=False)
display(missing_ratio.head(20).rename("missing_ratio").to_frame())

Jeu tabulaire agrégé construit.
Shape: (42, 11)


,row_id,subject_id,HMDPosY__mean,HMDPosY__std,HMDPosY__min,HMDPosY__max,HMDPosY__median,HMDPosY__q25,HMDPosY__q75,HMDPosY__slope,n_valid_features
0,0,136KD7,25.217463,112.593457,-78.71090,839.994,0.99000,-0.102315,3.731899,0.000036,1
1,1,0933,23.773801,112.145492,-79.72850,839.990,0.56000,-0.222960,2.831965,0.000034,1
2,2,7180bt,27.697605,119.945980,-87.57958,840.009,0.25000,-1.000000,3.139500,0.000047,1
3,3,22009611,28.882818,121.501576,-48.76670,840.027,0.38751,-0.295000,3.580814,0.000058,1
4,4,ab0710,24.423545,112.359693,-38.36740,839.990,0.94000,-0.207610,3.666630,0.000035,1


,missing_ratio
row_id,0.0
subject_id,0.0
HMDPosY__mean,0.0
HMDPosY__std,0.0
HMDPosY__min,0.0
HMDPosY__max,0.0
HMDPosY__median,0.0
HMDPosY__q25,0.0
HMDPosY__q75,0.0
HMDPosY__slope,0.0


: 